# SD 인페인팅 + LoRA — VisA PCB1–3 K장 생성 (Kaggle)

Colab 노트북(`colab/sd_lora_kshot.ipynb`)과 같은 작업을 Kaggle에서 합니다. 코드·결과 형식은 똑같습니다.

**처음 한 번 설정 (오른쪽 패널 → Settings)**
1. **Accelerator → GPU T4 x2** (P100은 최신 PyTorch가 지원하지 않습니다)
2. **Internet → On** (휴대폰 인증이 된 계정만 켤 수 있습니다)

그다음 **Run All**. 브라우저를 닫아도 돌게 하려면 오른쪽 위 **Save Version → Save & Run All (Commit)** 을 고르세요(최대 12시간).

| 단계 | 내용 |
|---|---|
| 데이터 | VisA 공식 tar에서 pcb1–3만 받아 옴. 데이터를 Kaggle에 따로 올릴 필요 없음 |
| 학습 | PCB별 LoRA 1개. GPU가 2장이면 PCB를 나눠 동시에 학습 |
| 생성 | PCB 3 × 종류 4 × 40장 = 480장, 누수 검사까지 자동 |

- 결과는 `/kaggle/working/sd_lora/`에 저장되고, 끝나면 노트북 **Output** 탭에서 zip으로 받을 수 있습니다.
- **이어서 하기**: 다른 세션에서 만든 결과(이 노트북의 이전 Output, 또는 Colab Drive의 `DL_TP/runs/sd_lora/lora` 폴더를 Kaggle Dataset으로 올린 것)를 **Add Input**으로 붙이면 이미 끝난 LoRA·조건은 건너뜁니다.
- 코드 설명: `Augmentation/sd_lora/README.md`

In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, sys, threading

REPO_URL = 'https://github.com/Junho-YEO-maker/26_2_DeepLearning.git'
BRANCH = 'main'                       # 특정 커밋으로 고정하려면 40자리 커밋 해시를 넣는다
REPO = Path('/tmp/26_2_DeepLearning')  # 코드는 Output에 섞이지 않게 /tmp에
AUG = REPO / 'Augmentation'
OUT = Path('/kaggle/working/sd_lora')  # 결과 (세션이 끝나면 Output으로 저장됨)
INPUT = Path('/kaggle/input')
K_VALUES = (1,)                       # 파일럿 후 (1, 2, 4)
SPLIT_SEED = 0
PER_TYPE = 40
STEPS = 1000                          # LoRA 학습 step

subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'GPU가 꺼져 있습니다: Settings → Accelerator → GPU T4 x2'
cap = torch.cuda.get_device_capability(0)
assert cap >= (7, 0), f'{torch.cuda.get_device_name(0)}(sm_{cap[0]}{cap[1]})는 지원하지 않습니다: Settings → Accelerator → GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('GPU', N_GPU, '장:', torch.cuda.get_device_name(0))

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)   # 실패하면 Settings → Internet → On
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
target = BRANCH if len(BRANCH) == 40 else f'origin/{BRANCH}'
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', target], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('코드 커밋:', COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'diffusers>=0.30', 'peft>=0.12', 'accelerate>=0.33'],
               check=True)
# 미리 깔린 오래된 torchao 때문에 최신 peft가 LoRA를 만들다 멈출 수 있다. 우리는 torchao를 안 쓰므로 지운다.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
OUT.mkdir(parents=True, exist_ok=True)
with (OUT / 'commits.txt').open('a') as f:
    f.write(COMMIT + '\n')
os.chdir(AUG)
sys.path.insert(0, str(AUG))


def _pump(proc, prefix):
    for line in proc.stdout:
        print(prefix + line, end='', flush=True)


def run(*args, gpus=None):
    # 하위 프로세스 출력을 셀에 그대로 보여 준다. gpus=[[인자...], [인자...]]면 GPU마다 하나씩 동시에 돌린다.
    jobs = [list(args)] if gpus is None else gpus
    procs = []
    for i, job in enumerate(jobs):
        env = dict(os.environ, PYTHONUNBUFFERED='1')
        if gpus is not None:
            env['CUDA_VISIBLE_DEVICES'] = str(i)
        proc = subprocess.Popen([sys.executable, '-u', *map(str, job)], stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, env=env)
        t = threading.Thread(target=_pump, args=(proc, f'[GPU{i}] ' if gpus is not None else ''))
        t.start()
        procs.append((proc, t))
    codes = []
    for proc, t in procs:
        codes.append(proc.wait())
        t.join()
    if any(codes):
        raise RuntimeError(f'실패 (exit {codes}): 위 출력의 마지막 오류 메시지를 확인하세요')

### 원본 데이터
Add Input으로 붙인 데이터 중에 `pcb1/image_anno.csv`가 있는 폴더가 있으면 그것을 쓰고, 없으면 VisA 공식 tar(약 1.8GB)에서 pcb1–3만 풀어 옵니다(수 분).
어느 쪽이든 쓰는 이미지는 열 때마다 분할표의 해시와 대조하고, test/val 이미지는 열 수 없게 막혀 있습니다.

In [ ]:
import tarfile, urllib.request

VISA_URL = 'https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar'
BOARDS = ('pcb1', 'pcb2', 'pcb3')
TYPES = ('bent', 'melt', 'scratch', 'missing')
LOCAL_DATA = Path('/tmp/visa_pcb123')


def find_attached():
    if not INPUT.is_dir():
        return None
    for root, dirs, files in os.walk(INPUT):
        if all((Path(root) / p / 'image_anno.csv').is_file() for p in BOARDS):
            return Path(root)
        if len(Path(root).relative_to(INPUT).parts) >= 5:
            dirs[:] = []
    return None


def download_visa(dest):
    print('VisA 공식 tar에서 pcb1–3 받는 중 (수 분)...', flush=True)
    pat = re.compile(r'(?:^|/)((?:pcb1|pcb2|pcb3)/.*)$')
    n = 0
    with urllib.request.urlopen(VISA_URL) as resp, tarfile.open(fileobj=resp, mode='r|') as tar:
        for m in tar:
            hit = pat.search(m.name)
            if not hit or not m.isfile():
                continue
            out = dest / hit.group(1)
            out.parent.mkdir(parents=True, exist_ok=True)
            with tar.extractfile(m) as src, open(out, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            n += 1
    print('파일', n, '개')


if all((LOCAL_DATA / p / 'image_anno.csv').is_file() for p in BOARDS):
    DATA = LOCAL_DATA
elif (DATA := find_attached()) is not None:
    print('붙인 데이터 사용:', DATA)
else:
    download_visa(LOCAL_DATA)
    DATA = LOCAL_DATA
for p in BOARDS:
    n = sum(1 for _ in (DATA / p / 'Data/Images').rglob('*.JPG'))
    print(p, n, '장')

from common.kshot import KShot
for k in K_VALUES:
    KShot(k, SPLIT_SEED, DATA)
print('원본 데이터:', DATA, '| 분할표 확인 완료:', K_VALUES)

### 이전 결과 이어받기 (있으면)
Add Input으로 붙인 것 중 `K{k}_seed{s}/pcb*/pytorch_lora_weights.safetensors`(LoRA)와 `generated/sd_lora/K{k}_seed{s}/manifest.csv`(생성 결과)를 찾아 `/kaggle/working/sd_lora`로 복사합니다. 없으면 아무것도 하지 않습니다.

In [ ]:
def restore_previous():
    if not INPUT.is_dir():
        return
    for w in INPUT.rglob('pytorch_lora_weights.safetensors'):
        d = w.parent
        if re.fullmatch(r'pcb[123]', d.name) and re.fullmatch(r'K\d+_seed\d+', d.parent.name):
            dest = OUT / 'lora' / d.parent.name / d.name
            if not (dest / w.name).is_file():
                shutil.copytree(d, dest, dirs_exist_ok=True)
                print('LoRA 이어받음:', d.parent.name, d.name)
    for m in INPUT.rglob('generated/sd_lora/K*_seed*/manifest.csv'):
        dest = OUT / 'Dataset/generated/sd_lora' / m.parent.name
        if not (dest / 'manifest.csv').is_file():
            shutil.copytree(m.parent, dest, dirs_exist_ok=True)
            print('생성 결과 이어받음:', m.parent.name)

restore_previous()

### 학습
PCB마다 LoRA 1개. 이미 있는 LoRA는 건너뜁니다. GPU가 2장이면 `[GPU0] pcb1, pcb3` / `[GPU1] pcb2`로 나눠 동시에 돌립니다.

In [ ]:
import json
for k in K_VALUES:
    args = ['-m', 'sd_lora.train', '--k', k, '--seed', SPLIT_SEED, '--steps', STEPS,
            '--data-root', DATA, '--runs-dir', OUT / 'lora']
    if N_GPU >= 2:
        run(gpus=[args + ['--pcb', 'pcb1', 'pcb3'], args + ['--pcb', 'pcb2']])
    else:
        run(*args)
    for pcb in BOARDS:
        info = json.loads((OUT / f'lora/K{k}_seed{SPLIT_SEED}/{pcb}/train.json').read_text())
        print(f'K{k} {pcb}: {info["minutes"]:.1f}분, 마지막 loss {info["final_loss"]:.4f}, {info["gpu"]}')

### 생성
조건마다 480장. 이미 manifest가 있으면 건너뜁니다. 끝나면 장수·누수 검사 후 `/kaggle/working/sd_lora/`로 옮기고 zip을 만듭니다.

In [ ]:
from collections import Counter
from common.kshot import check_manifest, read_rows

def publish(k):
    src = REPO / f'Dataset/generated/sd_lora/K{k}_seed{SPLIT_SEED}'
    rows = read_rows(src / 'manifest.csv')
    counts = Counter((r['category'], r['defect_type']) for r in rows)
    assert counts == {(p, t): PER_TYPE for p in BOARDS for t in TYPES}, counts
    print(check_manifest(src / 'manifest.csv'))
    dest = OUT / src.relative_to(REPO)
    shutil.copytree(src, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns('manifest.csv'))
    shutil.copyfile(src / 'manifest.csv', dest / 'manifest.csv.tmp')
    (dest / 'manifest.csv.tmp').replace(dest / 'manifest.csv')     # manifest는 마지막에: 있으면 완료된 것
    print('저장 완료:', dest, len(rows), '장')

for k in K_VALUES:
    name = f'K{k}_seed{SPLIT_SEED}'
    if (OUT / f'Dataset/generated/sd_lora/{name}/manifest.csv').is_file():
        print(f'[건너뜀] {name} 이미 있음')
    else:
        run('-m', 'sd_lora.generate', '--k', k, '--seed', SPLIT_SEED, '--per-type', PER_TYPE,
            '--data-root', DATA, '--runs-dir', OUT / 'lora', '--force')
        publish(k)
    # 받기 쉽게 zip: 압축을 풀면 Dataset/generated/sd_lora/K{k}_seed{s}/ 가 나온다 (manifest 상대경로와 일치)
    zip_path = shutil.make_archive(f'/kaggle/working/sd_lora_{name}', 'zip', root_dir=OUT,
                                   base_dir=f'Dataset/generated/sd_lora/{name}')
    print('zip:', zip_path, f'{os.path.getsize(zip_path) / 2**20:.0f} MB')

### 눈으로 확인
PCB·종류마다 한 장씩 결함 주변을 확대해서 보여 줍니다(빨간 테두리 = 마스크). 결함이 실제로 그려졌는지, 기판의 맞는 자리인지 확인하세요.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

k = K_VALUES[0]
rows = read_rows(OUT / f'Dataset/generated/sd_lora/K{k}_seed{SPLIT_SEED}/manifest.csv')
tiles = []
for pcb in BOARDS:
    for t in TYPES:
        r = next(x for x in rows if x['category'] == pcb and x['defect_type'] == t)
        im = Image.open(OUT / r['image']).convert('RGB')
        m = np.asarray(Image.open(OUT / r['mask'])) > 0
        ys, xs = np.where(m)
        cx, cy = (xs.min() + xs.max()) // 2, (ys.min() + ys.max()) // 2
        half = max(128, (max(np.ptp(xs), np.ptp(ys)) + 60) // 2)
        box = (cx - half, cy - half, cx + half, cy + half)
        crop = im.crop(box)
        ImageDraw.Draw(crop).rectangle([xs.min() - box[0] - 4, ys.min() - box[1] - 4,
                                        xs.max() - box[0] + 4, ys.max() - box[1] + 4], outline='red', width=3)
        crop = crop.resize((256, 256))
        ImageDraw.Draw(crop).text((4, 4), f'{pcb} {t}', fill='yellow')
        tiles.append(crop)
grid = Image.new('RGB', (256 * 4, 256 * 3))
for i, tile in enumerate(tiles):
    grid.paste(tile, ((i % 4) * 256, (i // 4) * 256))
grid.save('/kaggle/working/preview.png')
display(grid)

### 팀에 넘기기
- 노트북 **Output** 탭에서 `sd_lora_K{k}_seed0.zip`을 받아 Drive `DL_TP/runs/sd_lora/`에 풀고 폴더 공유 링크를 AD 팀에 전달
- manifest: zip 안의 `manifest.csv`를 저장소 `Dataset/generated/sd_lora/K{k}_seed0/manifest.csv`로 커밋
- LoRA(`sd_lora/lora/`)도 Output에 남으므로, 다음 세션에서 이 노트북 Output을 Add Input으로 붙이면 학습을 건너뜁니다